# Similarity-aware Label Smoothing

In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import CifarResNET18, CifarDenseNET121, CifarWideResNET2810
from metrics import top_label_ece, nll_loss
import random
import os
from scipy.stats import spearmanr, wilcoxon



## Hyperparams

In [42]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [43]:
dataset = "cifar100"
batch_size = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.1
epochs = 200
num_classes = int(dataset.removeprefix("cifar"))
epsilon = 0.005

## Training Utils

In [44]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [45]:
path = f"scores/4_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=5, tau=1.2):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(-class_dists / tau, dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [46]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [47]:
classwise_target = similarity_aware_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target[0])
print(classwise_target.shape)

# classwise_target = torch.eye(n=num_classes, device=device)
# print(classwise_target)
# print(classwise_target.shape)


tensor([9.9500e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 9.4294e-04, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 9.3770e-04, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 8.5326e-04,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
        0.0000e+00, 0.0000e+00, 0.0000e+

In [48]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

In [49]:
model = CifarResNET18(num_classes=num_classes).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=200
)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
)

Epoch 1/200 | Loss: 3.8181 | Test Acc: 0.1574 | Top-5 Test Acc: 0.4193


Epoch 2/200 | Loss: 3.0166 | Test Acc: 0.2537 | Top-5 Test Acc: 0.5553


Epoch 3/200 | Loss: 2.4310 | Test Acc: 0.3816 | Top-5 Test Acc: 0.7141


Epoch 4/200 | Loss: 2.0595 | Test Acc: 0.4299 | Top-5 Test Acc: 0.7539


Epoch 5/200 | Loss: 1.8335 | Test Acc: 0.4746 | Top-5 Test Acc: 0.7829


...

Epoch 196/200 | Loss: 0.0526 | Test Acc: 0.7876 | Top-5 Test Acc: 0.9388


Epoch 197/200 | Loss: 0.0528 | Test Acc: 0.7886 | Top-5 Test Acc: 0.9388


Epoch 198/200 | Loss: 0.0525 | Test Acc: 0.7887 | Top-5 Test Acc: 0.9363


Epoch 199/200 | Loss: 0.0526 | Test Acc: 0.7878 | Top-5 Test Acc: 0.9378


Epoch 200/200 | Loss: 0.0525 | Test Acc: 0.7895 | Top-5 Test Acc: 0.9381


In [50]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        logits_list.append(logits)
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)
print()
print("ECE:", top_label_ece(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))
test_acc = accuracy(model, test_loader)
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")
print()



ECE: 0.03993847966194153
NLL: 0.8626939654350281
Top-1 Test Acc: 0.7895 | Top-5 Test Acc: 0.9381



In [51]:
PATH = f"vae4_{'0'+str(epsilon).removeprefix("0.")}_{seed}.pth"
torch.save(model.state_dict(), PATH)